# Task 1: Design an In-Silico Perturbation Workflow

Develop a workflow to simulate knock-up and knock-down experiments for specified
genes. The workflow should provide flexibility for scaling to multiple genes.

## Strategy

GeneFormer tokenizes cells by **rank-ordering** genes by expression. We exploit
this directly: to simulate a perturbation we modify raw counts in `.X` before
passing data to the model, the rank shift propagates automatically.

| Perturbation | Operation | Effect on rank |
|---|---|---|
| **Knock-down** | Set expression → 0 | Gene moves to bottom of ranking |
| **Knock-up** | Set expression → per-cell max + 1 | Gene moves to top of ranking |

## Implementation:

The perturbation functions are implemented in `src/perturbation.py` rather than
directly in this notebook. This allows them to be imported consistently across
all tasks without duplicating code or depending on notebook execution order.

The following function is exposed:

- **`perturb_expression(adata, genes, direction, cell_mask=None)`**
  Perturbs one or more genes in a single AnnData object. Returns a new AnnData
  with the modified expression matrix, leaving the original untouched.
  An optional `cell_mask` restricts the perturbation to a subset of cells.

## Demo scope

To keep this notebook fast, we work with a small sample: 20 randomly selected
sALS cells from a single cell type (`Ex.L5.VAT1L_THSD4`). The full-scale
application across all ALS genes and cell types is in Task 2.

In [ ]:
# Imports
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy.sparse as sp

from src.data import load_data
from src.perturbation import perturb_expression

In [ ]:
# Load data and sample 20 sALS cells from one cell type
adata = load_data()

DEMO_CELLTYPE = "Ex.L5.VAT1L_THSD4"
mask = (
    (adata.obs["Group"] == "SALS") &
    (adata.obs["Cellstates_LVL3"] == DEMO_CELLTYPE)
).values

rng = np.random.default_rng(42)
idx = rng.choice(np.where(mask)[0], size=20, replace=False)
adata_demo = adata[idx].copy()
print(f"Demo subset: {adata_demo.n_obs} cells  |  cell type: {DEMO_CELLTYPE}")

In [ ]:
# Single-gene perturbation: MYT1L
TEST_GENE = "MYT1L"

kd = perturb_expression(adata_demo, genes=[TEST_GENE], direction="down")
ku = perturb_expression(adata_demo, genes=[TEST_GENE], direction="up")

g_idx = adata_demo.var_names.get_loc(TEST_GENE)

def col_mean(ann, idx):
    col = ann.X[:, idx]
    return col.toarray().mean() if sp.issparse(col) else col.mean()

print(f"\n{TEST_GENE} mean expression:")
print(f"  original:  {col_mean(adata_demo, g_idx):.3f}")
print(f"  knockdown: {col_mean(kd, g_idx):.3f}")
print(f"  knockup:   {col_mean(ku, g_idx):.3f}")

In [ ]:
# Multi-gene perturbation: pass a list of genes to perturb_expression
# to shift all of them simultaneously in a single call
BATCH_GENES = ["MYT1L", "REST", "SREBF2"]
available = [g for g in BATCH_GENES if g in adata_demo.var_names]

kd_multi = perturb_expression(adata_demo, genes=available, direction="down")
print(f"\nMulti-gene knock-down of {available}: shape {kd_multi.shape}")
print(f"Perturbation metadata: {kd_multi.uns['perturbation']}")